# Plan e.A -- Model A: Static OLS (Literal-Thesis Specification)

Specifies and estimates Model A exactly as written in the thesis's methodology chapter
(Ch. 3.5, Eq. 3.1) -- the fixed baseline every other result in this project (Model B, the
EViews cross-check, the "what would a naive reader conclude" narrative) is compared against.
Model A is estimated **unconditionally**: regardless of what step iii/v's stationarity testing
and decision branch find, it is always reported.

See `docs/2_plan/modeling/a_model_a_static_ols.md` for the full spec. This notebook is a
standalone, literal execution of that spec -- independent of `vi_estimation.ipynb`, which also
estimates Model A as part of a larger execution checklist alongside Model B. Both notebooks
implement the identical Model A formula and write to the same output files; if they ever
disagree, the plan doc (not either notebook) is authoritative on what the model is.

**Input** (`outputs/`):
- `analysis_frame_1990_2024.csv` (from step i -- the 1990-2024 analysis frame, N=35)
- `modeling_path_decision.csv` (from step v -- branch and per-variable I(0)/I(1)/I(2)
  classification, used only to determine whether the step-7 spurious-regression caveat applies)

**Outputs** (`outputs/`):
- `model_a_static_ols_coefficients.csv` -- coefficient, SE, t-stat, p-value, significance stars
- `model_a_static_ols_fit_stats.csv` -- R^2, adjusted R^2, F-statistic, F p-value, N, and the
  spurious-regression caveat flag/text


In [1]:
from pathlib import Path

import pandas as pd
import statsmodels.api as sm
from statsmodels.tools import add_constant

ROOT = Path.cwd().resolve().parents[1]
OUTPUT_DIR = ROOT / "outputs"

FRAME_IN = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
DECISION_IN = OUTPUT_DIR / "modeling_path_decision.csv"

COEF_OUT = OUTPUT_DIR / "model_a_static_ols_coefficients.csv"
FIT_STATS_OUT = OUTPUT_DIR / "model_a_static_ols_fit_stats.csv"

SIGNIFICANCE_LEVELS = [(0.01, "***"), (0.05, "**"), (0.10, "*")]


def stars(p_value: float) -> str:
    for threshold, mark in SIGNIFICANCE_LEVELS:
        if p_value < threshold:
            return mark
    return ""


# Model specification (thesis Ch. 3.5, Eq. 3.1):
# ERI = beta0 + beta1*DIVP + beta2*DIVM + beta3*INF + beta4*EXR + beta5*log(FDI) + beta6*SHOCK + eps
REGRESSORS = {
    "DIVP": "divp",
    "DIVM": "divm",
    "INF": "inflation_rate_pct",
    "EXR": "exchange_rate",
    "log(FDI)": "log_fdi",
    "SHOCK": "shock",
}

EXPECTED_SIGN = {
    "DIVP": "+ (H1)",
    "DIVM": "+ (H2)",
    "INF": "ambiguous a priori",
    "EXR": "ambiguous a priori",
    "log(FDI)": "+",
    "SHOCK": "-",
}


## Step 1 -- Confirm the analysis frame is available with all seven columns

In [2]:
frame = pd.read_csv(FRAME_IN).set_index("year")
assert frame.shape[0] == 35, f"expected 35-row analysis frame (1990-2024), got {frame.shape[0]}"

required_cols = ["eri"] + list(REGRESSORS.values())
missing = [c for c in required_cols if c not in frame.columns]
assert not missing, f"analysis frame is missing required columns: {missing}"

print(f"frame: {frame.shape}, years {frame.index.min()}-{frame.index.max()}")
frame[required_cols].head()


frame: (35, 10), years 1990-2024


,eri,divp,divm,inflation_rate_pct,exchange_rate,log_fdi,shock
year,,,,,,,
1990,0.3837,0.794432,0.830069,21.495250,40.06292,17.584935,0
1991,0.3678,0.766776,0.811653,12.185630,41.37150,17.693960,0
1992,0.3391,0.723811,0.737246,11.383440,43.82963,18.624648,0
1993,0.5452,0.719521,0.737053,11.746740,48.32217,19.085835,0
1994,0.6214,0.731781,0.729237,8.448712,49.41514,18.929983,0


## Model specification

```
ERI = beta0 + beta1*DIVP + beta2*DIVM + beta3*INF + beta4*EXR + beta5*log(FDI) + beta6*SHOCK + eps
```

| Symbol | Variable | Column | Role | Hypothesis | Expected sign |
|---|---|---|---|---|---|
| ERI | Economic Resilience Index | `eri` | Dependent | -- | -- |
| DIVP | Product diversification (HHI-based) | `divp` | Regressor | H1 | + |
| DIVM | Market diversification (HHI-based) | `divm` | Regressor | H2 | + |
| INF | Inflation rate | `inflation_rate_pct` | Control | -- | ambiguous a priori |
| EXR | Exchange rate (LKR/USD) | `exchange_rate` | Control | -- | ambiguous a priori |
| log(FDI) | Log of FDI net inflows | `log_fdi` | Control | -- | + |
| SHOCK | Crisis-year dummy (2008, 2009, 2020-22) | `shock` | Control | -- | - |

`log_fdi`, not raw FDI, is used here per the step-i deviation (flagged explicitly -- see
`docs/2_plan/modeling/c_shared_requirements_for_both_models.md`). The raw-FDI version is a
separate robustness check (step viii), never a substitute for this primary specification.

Estimated on the full 1990-2024 analysis frame (N=35), with a constant.

In [3]:
eri = frame["eri"].rename("ERI")
X = frame[list(REGRESSORS.values())].rename(columns={v: k for k, v in REGRESSORS.items()})
design = add_constant(X, has_constant="add")
design.head()


,const,DIVP,DIVM,INF,EXR,log(FDI),SHOCK
year,,,,,,,
1990,1.0,0.794432,0.830069,21.495250,40.06292,17.584935,0
1991,1.0,0.766776,0.811653,12.185630,41.37150,17.693960,0
1992,1.0,0.723811,0.737246,11.383440,43.82963,18.624648,0
1993,1.0,0.719521,0.737053,11.746740,48.32217,19.085835,0
1994,1.0,0.731781,0.729237,8.448712,49.41514,18.929983,0


## Step 2 -- Estimate via `statsmodels.api.OLS`

This is the exact package/function the user cross-checks against EViews' `LS` command.

In [4]:
model_a = sm.OLS(eri, design).fit()
print("Estimated with: statsmodels.api.OLS (constant added via statsmodels.tools.add_constant)")
print(model_a.summary())


Estimated with: statsmodels.api.OLS (constant added via statsmodels.tools.add_constant)
                            OLS Regression Results                            
Dep. Variable:                    ERI   R-squared:                       0.598
Model:                            OLS   Adj. R-squared:                  0.512
Method:                 Least Squares   F-statistic:                     6.953
Date:                Fri, 17 Jul 2026   Prob (F-statistic):           0.000133
Time:                        23:10:40   Log-Likelihood:                 28.688
No. Observations:                  35   AIC:                            -43.38
Df Residuals:                      28   BIC:                            -32.49
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------

## Steps 3-4 -- Full summary table, significance-starred

Coefficient, standard error, t-statistic, p-value per regressor; significance stars at the
1%/5%/10% levels (`***`/`**`/`*`).

In [5]:
model_a_coefs = pd.DataFrame({
    "term": model_a.params.index,
    "coef": model_a.params.values,
    "std_err": model_a.bse.values,
    "t_stat": model_a.tvalues.values,
    "p_value": model_a.pvalues.values,
})
model_a_coefs["significance"] = model_a_coefs["p_value"].map(stars)
model_a_coefs.to_csv(COEF_OUT, index=False)
print(f"Written -> {COEF_OUT}")
model_a_coefs


Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_a_static_ols_coefficients.csv


,term,coef,std_err,t_stat,p_value,significance
0,const,-2.334482,0.730522,-3.195634,0.003443,***
1,DIVP,1.268049,0.602285,2.105398,0.044354,**
2,DIVM,-0.805481,0.887658,-0.907423,0.371927,
3,INF,-0.004062,0.002733,-1.485997,0.148454,
4,EXR,-0.000941,0.000408,-2.307866,0.028610,**
5,log(FDI),0.142768,0.031814,4.487613,0.000112,***
6,SHOCK,-0.020333,0.068522,-0.296735,0.768859,


In [6]:
model_a_fit_stats = pd.DataFrame([{
    "package_function": "statsmodels.api.OLS",
    "n_obs": int(model_a.nobs),
    "r_squared": model_a.rsquared,
    "adj_r_squared": model_a.rsquared_adj,
    "f_statistic": model_a.fvalue,
    "f_pvalue": model_a.f_pvalue,
    "aic": model_a.aic,
    "bic": model_a.bic,
}])
model_a_fit_stats


,package_function,n_obs,r_squared,adj_r_squared,f_statistic,f_pvalue,aic,bic
0,statsmodels.api.OLS,35,0.598389,0.51233,6.953207,0.000133,-43.376734,-32.489297


## Step 5 -- Interpretation against H1 (DIVP) and H2 (DIVM)

For each hypothesis: sign vs. expected, statistical significance at 5%, and an
economic-significance translation (effect of a 0.1 increase in the regressor on ERI, holding
other variables constant).

In [7]:
def hypothesis_verdict(term, expected_sign, alpha=0.05):
    row = model_a_coefs.set_index("term").loc[term]
    coef, p_value = row["coef"], row["p_value"]
    sign_matches = (coef > 0) == (expected_sign == "+")
    if p_value < alpha and sign_matches:
        verdict = "SUPPORTED"
    elif p_value < alpha and not sign_matches:
        verdict = "REJECTED (significant, wrong sign)"
    else:
        verdict = "AMBIGUOUS (not significant at 5%)"
    delta_eri = coef * 0.1
    return coef, p_value, verdict, delta_eri


for label, hyp, expected_sign in [("DIVP", "H1", "+"), ("DIVM", "H2", "+")]:
    coef, p_value, verdict, delta_eri = hypothesis_verdict(label, expected_sign)
    print(f"{hyp} ({label}): coef={coef:.4f}, p={p_value:.4f} -> {verdict}")
    print(f"  Economic significance: a 0.1 increase in {label} is associated with a "
          f"{delta_eri:+.4f} change in ERI, holding other variables constant.")
    print()


H1 (DIVP): coef=1.2680, p=0.0444 -> SUPPORTED
  Economic significance: a 0.1 increase in DIVP is associated with a +0.1268 change in ERI, holding other variables constant.

H2 (DIVM): coef=-0.8055, p=0.3719 -> AMBIGUOUS (not significant at 5%)
  Economic significance: a 0.1 increase in DIVM is associated with a -0.0805 change in ERI, holding other variables constant.



## Step 6 -- Reported unconditionally

This model is estimated and reported in this notebook regardless of the branch outcome in
`docs/2_plan/analysis/v_decision_branch.md` (Branch A, B, or a halt at Branch C for a different
regressor). It is the fixed anchor for every comparison in this project -- Model B if triggered,
the EViews cross-check, and the Chapter 4 narrative.

## Step 7 -- Conditional validity caveat

If step iii's stationarity testing (via step v's decision, `modeling_path_decision.csv`) finds
any I(1) regressor, the caveat below is attached to the fit-stats table: estimates may reflect a
**spurious regression** -- significant coefficients driven by shared trends across
non-stationary series rather than a genuine relationship -- and should not be treated as the
primary basis for H1/H2 until compared against Model B's long-run coefficients. The caveat is
reported (or explicitly noted as not applicable) but never silently omitted.

In [8]:
decision = pd.read_csv(DECISION_IN).iloc[0]
i1_vars_all = [v.strip() for v in str(decision["i1_variables"]).split(",") if v.strip()]
i1_regressors = [r for r in REGRESSORS if r in i1_vars_all]

caveat_triggered = len(i1_regressors) > 0

if caveat_triggered:
    caveat_text = (
        f"SPURIOUS-REGRESSION CAVEAT: step v (branch {decision['branch']}) found the following "
        f"Model A regressor(s) to be I(1): {', '.join(i1_regressors)}. Model A's coefficients "
        "may reflect a spurious regression -- significant results driven by shared trends across "
        "non-stationary series rather than a genuine relationship. Do not treat these estimates "
        "as the primary basis for H1/H2 until compared against Model B's long-run coefficients."
    )
else:
    caveat_text = (
        "Not applicable: all six variables tested I(0) (or Branch A was taken), so no "
        "spurious-regression caveat applies to Model A."
    )

print(caveat_text)

model_a_fit_stats["spurious_regression_caveat_triggered"] = caveat_triggered
model_a_fit_stats["spurious_regression_caveat_text"] = caveat_text
model_a_fit_stats.to_csv(FIT_STATS_OUT, index=False)
print(f"\nWritten -> {FIT_STATS_OUT}")
model_a_fit_stats.T


SPURIOUS-REGRESSION CAVEAT: step v (branch B) found the following Model A regressor(s) to be I(1): DIVP, DIVM, EXR, log(FDI). Model A's coefficients may reflect a spurious regression -- significant results driven by shared trends across non-stationary series rather than a genuine relationship. Do not treat these estimates as the primary basis for H1/H2 until compared against Model B's long-run coefficients.

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/model_a_static_ols_fit_stats.csv


,0
package_function,statsmodels.api.OLS
n_obs,35
r_squared,0.598389
adj_r_squared,0.51233
f_statistic,6.953207
f_pvalue,0.000133
aic,-43.376734
bic,-32.489297
spurious_regression_caveat_triggered,True
spurious_regression_caveat_text,SPURIOUS-REGRESSION CAVEAT: step v (branch B) ...


## Formulas

- OLS estimator: `beta_hat = (X'X)^-1 X'y`.
- Standard errors, t-statistics, R^2, adjusted R^2, and F-statistic: standard OLS formulas as
  implemented by `statsmodels.api.OLS`.

## Decisions & flags

- Model A is always estimated -- settled in the requirements doc, not a judgment call made here.
- The spurious-regression caveat (step 7) is conditional on step v's output: stated when
  triggered, explicitly marked not-applicable (never silently omitted) otherwise.
- `log_fdi` vs. raw FDI: this specification uses `log_fdi` (flagged deviation, justified in step
  i); raw FDI is a robustness check only (step viii), never substituted into this primary
  specification.

## Definition of done

In [9]:
checks = {
    "Model A estimated on the full 35-obs frame with statsmodels.api.OLS, constant included":
        int(model_a.nobs) == 35 and "const" in model_a.params.index,
    "Full summary table reported with significance stars at 1%/5%/10%":
        set(model_a_coefs["significance"].unique()) <= {"", "*", "**", "***"},
    "Coefficients interpreted against H1/H2 in statistical and economic-significance terms":
        True,
    "Reported regardless of the analysis/v branch outcome":
        True,
    "Spurious-regression caveat attached whenever any regressor tested I(1)":
        "spurious_regression_caveat_triggered" in model_a_fit_stats.columns,
}

for description, passed in checks.items():
    print(("PASS" if passed else "FAIL") + f" -- {description}")

assert all(checks.values()), "Definition of done not fully met"


PASS -- Model A estimated on the full 35-obs frame with statsmodels.api.OLS, constant included
PASS -- Full summary table reported with significance stars at 1%/5%/10%
PASS -- Coefficients interpreted against H1/H2 in statistical and economic-significance terms
PASS -- Reported regardless of the analysis/v branch outcome
PASS -- Spurious-regression caveat attached whenever any regressor tested I(1)
